In [2]:
!pip install ultralytics

# Paste your Roboflow code under this cell

In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="gL9gSMJRQ9mmVYNytssf")
project = rf.workspace("infocomm-roboflow").project("distracted-driving-v2wk5-qdrrf")
version = project.version(2)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 27.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Distracted-Driving-2 in yolov8:: 100%|██████████| 44929/44929 [00:06<00:00, 7053.63it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
import os
import glob

for split in ['train', 'valid', 'test']:
    label_dir = f"{dataset.location}/{split}/labels"
    for label_file in glob.glob(f"{label_dir}/*.txt"):
        with open(label_file, 'r') as f:
            lines = f.readlines()

        classes = [int(line.strip().split()[0]) for line in lines if line.strip()]

        if 0 in classes and 1 in classes:
            lines = [line for line in lines if not line.strip().startswith('0 ')]

        with open(label_file, 'w') as f:
            f.writelines(lines)

print("fixed conflicting labels")


fixed conflicting labels


In [5]:
from ultralytics import YOLO

def freeze_layer(trainer):
    model = trainer.model
    num_freeze = 22
    print(f"Freezing {num_freeze} layers")
    freeze = [f'model.{x}.' for x in range(num_freeze)]  # layers to freeze
    for k, v in model.named_parameters():
        v.requires_grad = True  # train all layers
        if any(x in k for x in freeze):
            print(f'freezing {k}')
            v.requires_grad = False
    print(f"{num_freeze} layers are freezed.")
if __name__ == '__main__':
  model = YOLO('yolov8s.pt')
  model.train(
    data=f"/content/Distracted-Driving-2/data.yaml",
    epochs=67,
    batch=16,
    imgsz=640,
    freeze=0,
    optimizer='AdamW',
    lr0=0.0005,
    patience=15,
    cos_lr=True,
    cache=True,
    workers=4,
    name='MODEL'
)




Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/Distracted-Driving-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_binary_fast, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask

KeyboardInterrupt: 

In [ ]:
model = YOLO('/content/runs/detect/train_binary_fast-2/weights/best.pt')

In [ ]:
im = model.predict('/content/Distracted-Driving-2/test/images', save=True)

In [ ]:
PATH="/content/runs/detect/predict"
from google.colab.patches import cv2_imshow
import cv2
import os
test_images = os.listdir("/content/Distracted-Driving-2/test/images")
print(test_images[10:])  # show first 10 filenames
results = model.predict('/content/Distracted-Driving-2/test/images/img_35073_jpg.rf.b3072bc632ee3fc4a72d2915345f7cd3.jpg', save=True)
im = results[0].plot() # Get the annotated image (NumPy array) from the first Results object

cv2_imshow(im)

In [ ]:
from IPython.display import Image, display
display(Image(filename="/content/WhatsApp Image 2026-04-22 at 10.25.44.jpeg"))


In [ ]:
im = model.predict('/content/Distracted-Driving-2/test/images', save=True, conf=0.5)


In [ ]:
metrics = model.val(data="/content/Distracted-Driving-2/data.yaml", split='test', conf=0.25)


In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/detect/train_binary_fast-2/weights/best.pt')
metrics = model.val(data="/content/Distracted-Driving-2/data.yaml", split='val')

print(f"mAP@0.5:     {metrics.box.map50:.4f}")
print(f"mAP@0.5-95:  {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

for i, name in enumerate(metrics.names.values()):
    print(f"\n--- {name} ---")
    print(f"  Precision: {metrics.box.p[i]:.4f}")
    print(f"  Recall:    {metrics.box.r[i]:.4f}")
    print(f"  AP@0.5:    {metrics.box.ap50[i]:.4f}")

cm = metrics.confusion_matrix.matrix
print(f"\n--- Confusion Matrix ---")
for i, name in enumerate(metrics.names.values()):
    tp = int(cm[i][i])
    fp = int(sum(cm[:, i]) - cm[i][i])
    fn = int(sum(cm[i, :]) - cm[i][i])
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"{name}: TP={tp}, FP={fp}, FN={fn}, F1={f1:.4f}")
